In [18]:
import re
import json
# from pdf2image import convert_from_path
import fitz
import io
from PIL import Image
from paddleocr import PaddleOCR
import dotenv
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
import os

In [19]:
_ = load_dotenv(find_dotenv())
client = OpenAI(api_key = os.getenv("OPENAI_API_KEY"))

In [39]:
PDF_PATH = r"C:\Users\HP\Downloads\RAHIS 2510300207-Male55 years-94464.pdf"

In [13]:
ocr = PaddleOCR(use_angle_cls=True, lang="en")

[2026/01/16 18:04:39] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='C:\\Users\\HP/.paddleocr/whl\\det\\en\\en_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='C:\\Users\\HP/.paddleocr/whl\\rec\\en\\en_PP-OCRv4_rec_infer', rec_image_inverse=True, rec_image_shape='3, 48, 320', rec_batch_num=6, max_text_length=25

In [ ]:
def ocr_pdf_with_pymupdf(pdf_path):
    doc = fitz.open(pdf_path)
    all_lines = []

    for page_num in range(len(doc)):
        page = doc[page_num]

        pix = page.get_pixmap(dpi=300)
        img_bytes = pix.tobytes("png")

        image = Image.open(io.BytesIO(img_bytes))
        image_path = f"page_{page_num}.png"
        image.save(image_path)

        result = ocr.ocr(image_path)

        for block in result:
            for line in block:
                text = line[1][0].strip()
                if text:
                    all_lines.append(text)

    return all_lines

In [ ]:
"""
def parse_tests(lines):
    tests = []

    # pattern: TestName   value   range   unit
    value_pattern = re.compile(
        r"(?P<name>[A-Za-z0-9()./% \-]+)\s+"
        r"(?P<value>\d+\.?\d*)\s+"
        r"(?P<range>\d+\.?\d*\s*-\s*\d+\.?\d*|upto\s*\d+\.?\d*|<=?\s*\d+\.?\d*)?\s*"
        r"(?P<unit>mg/dL|g/dL|gm/dL|U/L|%|fL|pg|µIU/ml|lakh/cumm|/cumm)?",
        re.IGNORECASE
    )

    for line in lines:
        m = value_pattern.search(line)
        if m:
            tests.append({
                "name": m.group("name").strip(),
                "value": m.group("value"),
                "reference_range": (m.group("range") or "").strip(),
                "unit": (m.group("unit") or "").strip()
            })

    return tests
"""

In [22]:
"""
def parse_patient(lines):
    patient = {}

    for l in lines:
        if "Patient Name" in l:
            patient["name"] = l.split(":")[-1].strip()
        elif "Age / Gender" in l or "Age/Gender" in l:
            patient["age_gender"] = l.split(":")[-1].strip()
        elif "Patient ID" in l:
            patient["patient_id"] = l.split(":")[-1].strip()
        elif "Collection Time" in l:
            patient["collection_time"] = l.split(":")[-1].strip()
        elif "Reporting Time" in l:
            patient["reporting_time"] = l.split(":")[-1].strip()

    return patient
"""

'\ndef parse_patient(lines):\n    patient = {}\n\n    for l in lines:\n        if "Patient Name" in l:\n            patient["name"] = l.split(":")[-1].strip()\n        elif "Age / Gender" in l or "Age/Gender" in l:\n            patient["age_gender"] = l.split(":")[-1].strip()\n        elif "Patient ID" in l:\n            patient["patient_id"] = l.split(":")[-1].strip()\n        elif "Collection Time" in l:\n            patient["collection_time"] = l.split(":")[-1].strip()\n        elif "Reporting Time" in l:\n            patient["reporting_time"] = l.split(":")[-1].strip()\n\n    return patient\n'

Sending the extracted text to the open ai

In [36]:
def extract_lab_report_with_openai(text: str):

#     system_prompt = """
# You are a medical lab report extraction system.

# Rules:
# - Extract ONLY real patient info and lab tests
# - Ignore page numbers, footers, references, explanations, guidelines
# - Return ONLY valid JSON
# - If a field is missing, use empty string
# """
    system_prompt = """
You are a medical lab report extraction system.

CRITICAL RULES:

1. Some headings are PANEL NAMES (for example: "Differential Leukocyte Count (DLC)", "Liver Function Test", "Lipid Profile").
   These are NOT individual lab tests.

2. NEVER include panel names as tests.

3. Only include rows that have:
   - a biological analyte name (e.g., Neutrophils, Glucose, HbA1c)
   - a numeric value
   - optionally a unit and reference range.

4. Ignore:
   - section titles
   - panel names
   - page numbers
   - method lines (e.g., "Method: Microscopy")
   - interpretations
   - footnotes
   - references

5. Return ONLY valid JSON.
6. If a field is missing, use empty string.
7. If the patient name line contains a long number (8–12 digits), extract it as kmed_id.
    Example:
    "MS. PRIYANKA 2504280203"
    → name = "MS. PRIYANKA"
    → kmed_id = "2504280203"
"""

    user_prompt = f"""
Return JSON in this format:

{{
  "patient": {{
    "name": "",
    "kmed_Id": "",
    "age": "",
    "gender": "",
    "patient_id": "",
    "report_id": "",
    "collection_date": "",
    "report_date": ""
  }},
  "tests": [
    {{
      "name": "",
      "value": "",
      "unit": "",
      "reference_range": ""
    }}
  ]
}}

OCR TEXT:
{text}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
    )

    content = response.choices[0].message.content.strip()
    return json.loads(content)



In [37]:
"""
def main():
    print("Running OCR...")
    lines = ocr_pdf_with_pymupdf(PDF_PATH)

    print("Parsing patient info...")
    patient = parse_patient(lines)

    print("Parsing test values...")
    tests = parse_tests(lines)

    result = {
        "patient": patient,
        "tests": tests
    }

    print(json.dumps(result, indent=2))


if __name__ == "__main__":
    main()
    """

'\ndef main():\n    print("Running OCR...")\n    lines = ocr_pdf_with_pymupdf(PDF_PATH)\n\n    print("Parsing patient info...")\n    patient = parse_patient(lines)\n\n    print("Parsing test values...")\n    tests = parse_tests(lines)\n\n    result = {\n        "patient": patient,\n        "tests": tests\n    }\n\n    print(json.dumps(result, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n    '

In [40]:
def main():
    print("Running OCR...")
    lines = ocr_pdf_with_pymupdf(PDF_PATH)

    full_text = "\n".join(lines)

    print("Sending to OpenAI for structuring...")
    data = extract_lab_report_with_openai(full_text)

    print("\n===== FINAL STRUCTURED OUTPUT =====\n")
    print(json.dumps(data, indent=2))


if __name__ == "__main__":
    main()

Running OCR...
[2026/01/17 15:11:58] ppocr DEBUG: dt_boxes num : 108, elapsed : 0.8703827857971191
[2026/01/17 15:11:58] ppocr DEBUG: cls num  : 108, elapsed : 0.3852710723876953
[2026/01/17 15:12:05] ppocr DEBUG: rec_res num  : 108, elapsed : 6.269110918045044
[2026/01/17 15:12:06] ppocr DEBUG: dt_boxes num : 82, elapsed : 0.2348041534423828
[2026/01/17 15:12:06] ppocr DEBUG: cls num  : 82, elapsed : 0.3824763298034668
[2026/01/17 15:12:11] ppocr DEBUG: rec_res num  : 82, elapsed : 4.730190277099609
[2026/01/17 15:12:12] ppocr DEBUG: dt_boxes num : 72, elapsed : 0.3046436309814453
[2026/01/17 15:12:12] ppocr DEBUG: cls num  : 72, elapsed : 0.2773144245147705
[2026/01/17 15:12:17] ppocr DEBUG: rec_res num  : 72, elapsed : 4.5734052658081055
[2026/01/17 15:12:18] ppocr DEBUG: dt_boxes num : 42, elapsed : 0.32818174362182617
[2026/01/17 15:12:18] ppocr DEBUG: cls num  : 42, elapsed : 0.16567325592041016
[2026/01/17 15:12:22] ppocr DEBUG: rec_res num  : 42, elapsed : 3.9754083156585693
[2